# Segment-level phonological probing

This notebook builds up the real (segment-level) evaluation in steps:

1. **Extract phonological features** from the FLEURS transcriptions  ← *this notebook, for now*
2. Load embeddings + forced-align phonemes to frames
3. Probe each feature (within-language and cross-lingual)
4. Evaluate hypotheses H1–H3 one by one

**Step 1** also doubles as the validation harness for `src/phonology.py` — it confirms espeak-ng + panphon are installed and that the labels and the per-feature segment filters look sane.

In [1]:
import os, sys

# Run as if from the project root, regardless of where this notebook lives
if os.path.basename(os.getcwd()) == "notebook":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pickle
import numpy as np
import pandas as pd
from datasets import load_dataset

from src.phonology import (
    phonological_features, text_to_ipa, keep_for, FEATURES, SEGMENT_FILTERS,
)

pd.set_option("display.max_rows", 60)

/home/hpc/vlbi/vlbi108v/miniconda3/envs/feature/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
LANGUAGES = ["en_us", "de_de", "es_419"]
SPLIT = "train"
N_SAMPLES = 100          # per language; keep in sync with what you extracted embeddings for
OUTPUT = "artifacts/phon_features.pkl"
print(FEATURES, "| filters:", SEGMENT_FILTERS)

['voicing', 'nasal', 'manner', 'place'] | filters: {'voicing': 'obstruent', 'nasal': 'all', 'manner': 'consonant', 'place': 'consonant'}


## 1a. Sanity check

Phonemize one short phrase per language and show the per-phoneme labels. If this errors, the usual cause is a missing **espeak-ng** binary (`conda install -c conda-forge espeak-ng`) or `panphon`/`phonemizer` not installed in the env.

In [3]:
examples = {
    "en_us": "the quick brown fox jumps",
    "de_de": "der schnelle braune Fuchs",
    "es_419": "el rapido zorro marron",
}
for lang in LANGUAGES:
    text = examples[lang]
    print(f"\n=== {lang}: {text!r}")
    print("IPA:", text_to_ipa(text, lang))
    display(pd.DataFrame(phonological_features(text, lang)))


=== en_us: 'the quick brown fox jumps'


RuntimeError: espeak not installed on your system

## 1b. Extract over the dataset

Load the FLEURS transcriptions (audio is not decoded — we only read the `transcription`/`id` columns) and phonemize the first `N_SAMPLES` utterances per language into one long, tidy table: one row per phoneme.

In [ ]:
rows = []
for lang in LANGUAGES:
    ds = load_dataset("google/fleurs", lang, split=SPLIT, trust_remote_code=True)
    ds = ds.select(range(min(N_SAMPLES, len(ds))))
    for utt_id, text in zip(ds["id"], ds["transcription"]):
        for idx, seg in enumerate(phonological_features(text, lang)):
            rows.append({"language": lang, "utt_id": utt_id, "idx": idx, **seg})
    print(f"{lang}: done")

df = pd.DataFrame(rows)
print(f"\nTotal phonemes: {len(df)}")
df.head(15)

## 1c. Label distributions &amp; the segment-filter effect

Two things to check: (a) every feature has a usable class balance, and (b) the segment filters do their job. The clearest case is **voicing**: across *all* segments it collapses toward `voiced` (sonorants + vowels dominate) — the MVP's failure mode — but restricted to **obstruents** the voiced/voiceless contrast is balanced and probeable.

In [ ]:
print("voicing on ALL segments:")
print(df["voicing"].value_counts(normalize=True).round(3).to_string())
print("\nvoicing on OBSTRUENTS only (what we actually probe):")
print(df[df.is_obstruent]["voicing"].value_counts(normalize=True).round(3).to_string())

In [ ]:
for feat in FEATURES:
    mask = df.apply(lambda r: keep_for(feat, r), axis=1)
    sub = df[mask]
    print(f"\n=== {feat}  (filter: {SEGMENT_FILTERS[feat]}, n={len(sub)}) ===")
    print(sub.groupby("language")[feat].value_counts().to_string())

## 1d. Save

Persist the per-phoneme table for the next steps. (The probing pipeline in step 2 re-phonemizes per word during alignment; this saved table is for inspection and for reproducing the label distributions above.)

In [ ]:
os.makedirs(os.path.dirname(OUTPUT) or ".", exist_ok=True)
df.to_pickle(OUTPUT)
print(f"Saved {len(df)} phoneme rows to {OUTPUT}")

## Next (step 2)

Load the saved embeddings, forced-align each utterance with `src/align.py` (`get_aligner` → `align_words` → `phoneme_spans` → `segment_dataset`), and stack `(X, y)` per feature/layer/language. Then step 3 runs `evaluate_probe` / `cross_lingual_probe`.